In [2]:
import requests
import os
import pandas as pd
import json

from dotenv import load_dotenv
from agents import Agent, Runner, trace
from openai import OpenAI

from pydantic import BaseModel, Field, field_validator

import itertools
from tqdm import tqdm

In [3]:
import sys
sys.path.append("/Users/felipeformenti/dev/fformenti/nba_bets")

In [4]:
from src.config.paths import (
    TEAMS_CITIES_LOCATIONS_HISTORY_PROCESSED_PATH,
    LOCATIONS_DISTANCES_PATH,
)
from src.config.constants import SERPER_ENDPOINT

In [5]:
SERPER_API_KEY = os.getenv("SERPER_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")

load_dotenv(override=True)
openai = OpenAI()


In [6]:
SERPER_API_KEY


'e548fbc815acbf4ba9f0c1956bf9569ed31270fb'

In [ ]:
def make_google_distance_query(city1: str, city2: str):
    return f"What is the distance in miles from {city1} to {city2}?"

def google_call(query) -> float:
    payload = {"q": query}
    headers = {
        "X-API-KEY": SERPER_API_KEY,
        "Content-Type": "application/json",
    }

    response = requests.request("POST", SERPER_ENDPOINT, headers=headers, json=payload)
    return response.text

def parse_google_search(google_response) -> str:
    resp_json = json.loads(google_response)
    relevant_context = [i["snippet"] for i in resp_json["organic"][0:3]]
    result = "\n\n".join(relevant_context)
    return result

class distance(BaseModel):
    """Distance between two cities."""
    straight_line_distance: float = Field(
        description="The straight line distance between the two cities in miles"
    )
    driving_distance: float = Field(
        description="The driving distance between the two cities in miles"
    )

In [ ]:
teams_locations = pd.read_csv(TEAMS_CITIES_LOCATIONS_HISTORY_PROCESSED_PATH)
teams_locations["city_state"] = (
    teams_locations["city"] + ", " + teams_locations["state"]
)

In [ ]:
def get_distances_ai(google_query, google_results_txt):
    system_prompt = """
    Your task is to extract the distance between two cities. 
    The context may have both driving distance and straight line distance (sometimes referred as flight distance, shortest distance or airline distance). 
    Get both distances if they are available, if either is missing, return the None.
    Use the following context to answer the user's question:\n\n
    """

    users_prompt = f"""
    Use the following context to answer the user's question:
    {google_results_txt}

    Question:
    {google_query}
    """

    messages = [{"role": "system", "content": system_prompt}] + [
        {"role": "user", "content": users_prompt}
    ]

    model = "gpt-4.1-nano"
    response = openai.chat.completions.parse(
        model=model, messages=messages, response_format=distance
    )

    distance_dict = json.loads(response.choices[0].message.content)
    return distance_dict


In [ ]:
unique_locations = list(set(teams_locations["city_state"]))
locations_distances = []
for pair in tqdm(itertools.combinations(unique_locations, 2)):
    cities_dict = {"from":pair[0], "to": pair[1]}
    
    google_query = make_google_distance_query(pair[0], pair[1])
    google_response = google_call(google_query)
    google_results_txt = parse_google_search(google_response)

    distance_dict = get_distances_ai(google_query, google_results_txt)
    locations_distances.append({**cities_dict, **distance_dict})

locations_distances_df = pd.DataFrame(locations_distances)

In [ ]:
locations_distances_df.to_csv(LOCATIONS_DISTANCES_PATH, index=False)

In [6]:
locations_distances_df = pd.read_csv(LOCATIONS_DISTANCES_PATH)
teams_history_df = pd.read_csv(TEAMS_CITIES_LOCATIONS_HISTORY_PROCESSED_PATH)

In [8]:
locations_distances_df.head(1)

,from,to,straight_line_distance,driving_distance
0,"Sacramento, California","Houston, Texas",1608.0,1929.0


In [7]:
teams_history_df.head()

,teamId,teamCity,teamName,teamAbbrev,season,Conference,teamFullName,city,state
0,1610612737,Atlanta,Hawks,ATL,1968/69,East,Atlanta Hawks,Atlanta,Georgia
1,1610612737,Atlanta,Hawks,ATL,1969/70,East,Atlanta Hawks,Atlanta,Georgia
2,1610612737,Atlanta,Hawks,ATL,1970/71,East,Atlanta Hawks,Atlanta,Georgia
3,1610612737,Atlanta,Hawks,ATL,1971/72,East,Atlanta Hawks,Atlanta,Georgia
4,1610612737,Atlanta,Hawks,ATL,1972/73,East,Atlanta Hawks,Atlanta,Georgia


In [ ]:
locations_distances_df.merge(teams_locations, left_on="from", right_on="city_state", how="left")

In [ ]:

locations_distances_df.to_csv("locations_distances.csv", index=False)

In [ ]:
NBA_DATA_SCIENTIST_SYSTEM_PREFIX = """
You are a data scientist expert data cleaning and data wrangling.
I'm working in a data science project about NBA games.
You are here to help me assist me with my data cleaning and data wrangling tasks.
"""
